# 01 - Data Preparation

This notebook imports, cleans and transforms the raw index data.

# 1. General directory and data assignment

In [ ]:
# import necessary libraries
from pathlib import Path 
import sys
import xlrd

import pandas as pd 
import numpy as np 

#show versions of the libraries
print("Python version:", sys.version)
print("Pandas version:", pd.__version__)
print("Numpy version:", np.__version__)
print("xlrd version:", xlrd.__version__)

In [ ]:
# Get the current working directory
current_directory = Path.cwd()

print("Current working directory:", current_directory)

In [ ]:
# Check if the current working directory is the "notebooks" folder
if current_directory.name == "notebooks":
    repository_root = current_directory.parent
    print("Changed working directory to:", current_directory)
else: 
    repository_root = current_directory

print("Repository root directory:", repository_root)

In [ ]:
# Define the data directories
raw_data_directory = repository_root / "data" / "raw"
processed_data_directory = repository_root / "data" / "processed"
metadata_directory = repository_root / "data" / "metadata"

print("Raw data directory:", raw_data_directory)
print("Processed data directory:", processed_data_directory)
print("Metadata directory:", metadata_directory)

In [ ]:
# Check if the directories exist
print("Repository exists:", repository_root.exists())
print("Raw data directory exists:", raw_data_directory.exists())
print("Processed data directory exists:", processed_data_directory.exists())
print("Metadata directory exists:", metadata_directory.exists())

In [ ]:
# List all files in the raw data directory, watch for .xls files and exclude README.md, see how many files are there
raw_files = [
    path
    for path in raw_data_directory.iterdir()
    if path.is_file() and path.name != "README.md"
]

print(f"Number of raw data files: {len(raw_files)}")

for file_path in raw_files:
    print(file_path.name)

In [ ]:
# Filter the files to only include those with the ".xls" extension (my raw data files are in .xls format)
xls_files = [
    path
    for path in raw_files
    if path.suffix.lower() == ".xls"
]

print(f"Number of XLS files: {len(xls_files)}")

for index, file_path in enumerate(xls_files):
    print(index, file_path.name)

# 2. Define and test procedure with the first file

In [ ]:
# Select the first XLS file from the list
first_file = xls_files[0]

print("Selected file:", first_file.name)

In [ ]:
# Read the selected XLS files sheet name using pandas
excel_file = pd.ExcelFile(first_file)

print("Sheet names:", excel_file.sheet_names)

In [ ]:
# Select the first sheet from the Excel file
sheet_name = excel_file.sheet_names[0]

print("Selected sheet:", sheet_name)

In [ ]:
# Read the first 25 rows of the selected sheet to preview the data
top_preview = pd.read_excel(
    first_file,
    sheet_name=sheet_name,
    header=None,
    nrows=25
)

top_preview

In [ ]:
# Read the entire selected sheet
full_raw_sheet = pd.read_excel(
    first_file,
    sheet_name=sheet_name,
    header=None,
    engine="xlrd"
)

# Read the last 35 rows of the full raw sheet to preview the bottom of the data
bottom_preview = full_raw_sheet.tail(35)

bottom_preview

In [ ]:
# Get the number of rows and columns in the raw sheet
print("Number of rows:", full_raw_sheet.shape[0])
print("Number of columns:", full_raw_sheet.shape[1])

In [ ]:
# Define the date column and index-level column 
date_column = full_raw_sheet.columns[0]
index_level_column = full_raw_sheet.columns[1]

print("Date column:", date_column)
print("Index-level column:", index_level_column)

In [ ]:
selected_data = full_raw_sheet[
    [date_column, index_level_column]
].copy()

selected_data.columns = ["date", "index_level"]

print("Selected columns:", selected_data.columns.tolist())

print("Number of rows:", len(selected_data))

In [ ]:
clean_data = selected_data.copy()

# Convert the date column to pandas datetime values.
clean_data["date"] = pd.to_datetime(
    clean_data["date"],
    format="%b %d, %Y",
    errors="coerce"
)

# Remove US thousands separators before converting index levels to numbers.
clean_data["index_level"] = pd.to_numeric(
    clean_data["index_level"]
        .astype("string")
        .str.strip()
        .str.replace(",", "", regex=False),
    errors="coerce"
)

In [ ]:
# Remove rows with missing values and the copyright notice
rows_before_cleaning = len(clean_data)

clean_data = clean_data.dropna(
    subset=["date", "index_level"]
).copy()

rows_after_cleaning = len(clean_data)

print("Rows before cleaning:", rows_before_cleaning)
print("Rows after cleaning:", rows_after_cleaning)
print(
    "Removed non-data rows:",
    rows_before_cleaning - rows_after_cleaning
)

In [ ]:
# Sort the data by date and remove duplicate dates, keeping the last occurrence
clean_data = clean_data.sort_values("date", kind="stable")

clean_data = clean_data.drop_duplicates(
    subset="date",
    keep="last"
)

clean_data = clean_data.reset_index(drop=True)

In [ ]:
# Check the cleaned data for any remaining issues
print("Valid observations:", len(clean_data))
print("First date:", clean_data["date"].min())
print("Last date:", clean_data["date"].max())

print("Missing dates:", clean_data["date"].isna().sum())
print("Missing index levels:", clean_data["index_level"].isna().sum())
print("Duplicate dates:", clean_data["date"].duplicated().sum())
print("Non-finite index levels:", (~np.isfinite(clean_data["index_level"])).sum())
print(
    "Non-positive index levels:",
    (clean_data["index_level"] <= 0).sum()
)

In [ ]:
# Check if the data is monthly and if there are any months with multiple observations
clean_data["month"] = clean_data["date"].dt.to_period("M")

observations_per_month = clean_data.groupby("month").size()

print("Number of observed months:", len(observations_per_month))
print(
    "Months with multiple observations:",
    (observations_per_month > 1).sum()
)

In [ ]:
# Check for missing months in the data
expected_months = pd.period_range(
    start=clean_data["month"].min(),
    end=clean_data["month"].max(),
    freq="M"
)

missing_months = expected_months.difference(
    observations_per_month.index
)

print("Missing months:", len(missing_months))

In [ ]:
# Create a new DataFrame with monthly levels, using the last observation of each month
monthly_levels = clean_data[
    ["date", "index_level"]
].copy()

monthly_levels["month"] = monthly_levels["date"].dt.to_period("M")

monthly_levels = (
    monthly_levels
    .sort_values("date", kind="stable")
    .groupby("month", sort=True)["index_level"]
    .last()
    .to_frame(name="value")
)

monthly_levels.index = monthly_levels.index.to_timestamp("M")
monthly_levels.index.name = "date"

In [ ]:
# Check the structure of the monthly_levels DataFrame
print("DataFrame type:", type(monthly_levels))
print("Index type:", type(monthly_levels.index))
print("Column names:", monthly_levels.columns.tolist())

print("Number of monthly levels:", len(monthly_levels))
print("First month:", monthly_levels.index.min())
print("Last month:", monthly_levels.index.max())
print("Duplicate months:", monthly_levels.index.duplicated().sum())

In [ ]:
# Calculate monthly returns as percentage change of the monthly levels
monthly_returns = monthly_levels.pct_change(
    fill_method=None
)

monthly_returns = monthly_returns.dropna().copy()

In [ ]:
# Check for any issues in the monthly returns DataFrame (especially if there were unusual returns like -100% or more than 50% in absolute value)
print("Number of index levels:", len(monthly_levels))
print("Number of monthly returns:", len(monthly_returns))

print("Missing returns:", monthly_returns.isna().sum().sum())
print("Duplicate months:", monthly_returns.index.duplicated().sum())

print(
    "Returns less than or equal to -100%:",
    (monthly_returns <= -1).sum().sum()
)

print(
    "Absolute monthly returns above 50%:",
    (monthly_returns.abs() > 0.50).sum().sum()
)

In [ ]:
# More validation checks
assert len(missing_months) == 0
assert np.isfinite(monthly_levels.to_numpy()).all()
assert (monthly_levels > 0).all().all()
assert len(monthly_returns) == len(monthly_levels) - 1
assert not monthly_returns.isna().any().any()
assert not monthly_returns.index.duplicated().any()
assert np.isfinite(monthly_returns.to_numpy()).all()
assert (monthly_returns > -1).all().all()

print("All validation checks passed.")

# 3. Implement an import-function for all data files

In [ ]:
def load_msci_index(
        file_path,
        series_name,
        header_row=6,
        sheet_name=0
):
    
    # Load and data from an .xls file
    raw_data = pd.read_excel(
        file_path,
        sheet_name=sheet_name,
        header=header_row,
        engine="xlrd"
    )

    # Drop any columns that are completely empty
    raw_data = raw_data.dropna(
        axis="columns",
        how="all"
    )

    if raw_data.shape[1] < 2:
        raise ValueError(f"Expected at least two columns in {file_path}")

    # Strip whitespace from column names
    raw_data.columns = [
        str(column).strip() for column in raw_data.columns
    ]

    # Select the first two columns (date and index level) and rename them
    selected_data = raw_data.iloc[:, [0, 1]].copy()

    selected_data.columns = ["date", "index_level"]

    # Convert the date column to pandas datetime values and the index level column to numeric values, handling errors by coercing invalid entries to NaN
    selected_data["date"] = pd.to_datetime(
        selected_data["date"],
        format="%b %d, %Y",
        errors="coerce"
        )
    
    # Remove US thousands separators before converting index levels to numbers
    selected_data["index_level"] = pd.to_numeric(
        selected_data["index_level"]
            .astype("string")
            .str.strip()
            .str.replace(",", "", regex=False),
        errors="coerce"
        )

    # Drop any rows with missing values in the date or index level columns
    clean_data = selected_data.dropna(
        subset=["date", "index_level"]
    ).copy()

    if clean_data.empty:
        raise ValueError(f"No valid observations found in {file_path}")

    if not np.isfinite(clean_data["index_level"]).all():
        raise ValueError(f"Non-finite index levels found in {file_path}")

    if not (clean_data["index_level"] > 0).all():
        raise ValueError(f"Non-positive index levels found in {file_path}")

    clean_data = clean_data.sort_values("date", kind="stable")
    clean_data["month"] = clean_data["date"].dt.to_period("M")

    observed_months = pd.PeriodIndex(clean_data["month"].unique()).sort_values()
    expected_months = pd.period_range(
        start=observed_months.min(),
        end=observed_months.max(),
        freq="M"
    )
    missing_months = expected_months.difference(observed_months)

    if len(missing_months) > 0:
        raise ValueError(
            f"Missing months in {file_path}: {missing_months.astype(str).tolist()}"
        )

    clean_data = (
        clean_data
        .groupby("month", sort=True, as_index=False)
        .tail(1)
        .sort_values("month")
    )

    clean_data["date"] = clean_data["month"].dt.to_timestamp("M")
    clean_data = clean_data.set_index("date")

    index_series = clean_data["index_level"].copy()
    index_series.name = series_name

    if index_series.index.duplicated().any():
        raise ValueError(f"Duplicate months found in {file_path}")

    if not index_series.index.is_monotonic_increasing:
        raise ValueError(f"Dates are not sorted in {file_path}")

    return index_series

# 3.1 List and load the files

In [ ]:
# Load index metadata and raw file names
metadata_columns = [
    "series_id", "factor", "index_name", "index_code", "parent_index",
    "currency", "return_type", "frequency", "launch_date",
    "first_observation", "last_observation", "download_date",
    "raw_filename", "source_url", "backtested_history", "notes"
]
metadata_date_columns = [
    "launch_date", "first_observation", "last_observation", "download_date"
]

index_metadata = pd.read_csv(
    metadata_directory / "index_metadata.csv",
    parse_dates=metadata_date_columns
)

assert index_metadata.columns.tolist() == metadata_columns
assert index_metadata["series_id"].is_unique
assert index_metadata["raw_filename"].is_unique
assert (index_metadata["first_observation"] <= index_metadata["last_observation"]).all()
assert (index_metadata["last_observation"] <= index_metadata["download_date"]).all()
assert index_metadata["backtested_history"].eq(
    index_metadata["first_observation"] < index_metadata["launch_date"]
).all()

metadata_by_series = index_metadata.set_index("series_id")
factor_files = index_metadata.set_index("series_id")["raw_filename"].to_dict()

In [ ]:
# Test if the files are found
for series_name, filename in factor_files.items():
    file_path = raw_data_directory / filename

    print(
        series_name,
        "|",
        filename,
        "| exists:",
        file_path.exists()
    )

assert all((raw_data_directory / filename).exists() for filename in factor_files.values())

In [ ]:
# Read all files
index_series = {}

for series_name, filename in factor_files.items():

    file_path = raw_data_directory / filename

    series = load_msci_index(
        file_path=file_path,
        series_name=series_name,
        header_row=6,
        sheet_name=0
    )

    index_series[series_name] = series

    expected_first = metadata_by_series.loc[series_name, "first_observation"].to_period("M").to_timestamp("M")
    expected_last = metadata_by_series.loc[series_name, "last_observation"].to_period("M").to_timestamp("M")
    expected_months = pd.period_range(expected_first, expected_last, freq="M")

    assert series.index.min() == expected_first
    assert series.index.max() == expected_last
    assert series.index.to_period("M").equals(expected_months)
    assert np.isfinite(series.to_numpy()).all()
    assert (series > 0).all()

    # Check the structure of the each files data
    print(series_name, "| observations:", len(series), "| first date:", series.index.min(), "| last date:", series.index.max())



# 3.2 Connect the files data

In [ ]:
# Combine all series into a single DataFrame
monthly_levels = pd.concat(
    index_series.values(),
    axis="columns",
    join="outer"
)

monthly_levels = monthly_levels.sort_index()

monthly_levels.index.name = "date"

In [ ]:
# Check the structure of the combined DataFrame
print("Shape of combined DataFrame:", monthly_levels.shape)
print("Columns:", monthly_levels.columns.tolist())
print("First date:", monthly_levels.index.min())
print("Last date:", monthly_levels.index.max())
print("Duplicate months/dates:", monthly_levels.index.duplicated().sum())

In [ ]:
# Check factor availability (missing values)
availability = pd.DataFrame(
    {
        "observations": monthly_levels.notna().sum(),
        "missing_values": monthly_levels.isna().sum(),
        "first_observation": monthly_levels.apply(lambda series: series.first_valid_index()),
        "last observation": monthly_levels.apply(lambda series: series.last_valid_index())
    }
)

availability

# 3.3 Calculate returns

In [ ]:
monthly_returns_full = monthly_levels.pct_change(
    fill_method=None
)

In [ ]:
# Common time period for all factors (drop any rows with missing values)
monthly_returns_common = ( 
    monthly_returns_full.dropna(how="any").copy()
)

In [ ]:
# Check combined sample shapes

print("Full return dataset:", monthly_returns_full.shape)
print("Common return dataset:", monthly_returns_common.shape)
print("Common sample:", monthly_returns_common.index.min(), "to", monthly_returns_common.index.max())

In [ ]:
# More validation checks
observed_levels = monthly_levels.to_numpy(dtype=float, na_value=np.nan)
observed_levels = observed_levels[~np.isnan(observed_levels)]
expected_level_months = pd.period_range(
    monthly_levels.index.min(), monthly_levels.index.max(), freq="M"
)
expected_common_start = max(series.index.min() for series in index_series.values()) + pd.offsets.MonthEnd(1)
expected_common_end = min(series.index.max() for series in index_series.values())
expected_common_months = pd.period_range(
    expected_common_start, expected_common_end, freq="M"
)
observed_returns_full = monthly_returns_full.to_numpy(dtype=float, na_value=np.nan)
observed_returns_full = observed_returns_full[~np.isnan(observed_returns_full)]

assert monthly_levels.columns.tolist() == list(factor_files)
assert not monthly_levels.index.duplicated().any()
assert monthly_levels.index.to_period("M").equals(expected_level_months)
assert np.isfinite(observed_levels).all()
assert (observed_levels > 0).all()
assert np.isfinite(observed_returns_full).all()
assert (observed_returns_full > -1).all()
assert not monthly_returns_common.empty
assert not monthly_returns_common.isna().any().any()
assert not monthly_returns_common.index.duplicated().any()
assert monthly_returns_common.index.min() == expected_common_start
assert monthly_returns_common.index.max() == expected_common_end
assert monthly_returns_common.index.to_period("M").equals(expected_common_months)
assert np.isfinite(monthly_returns_common.to_numpy(dtype=float, na_value=np.nan)).all()
assert (monthly_returns_common > -1).all().all()

print("All return validation checks passed.")

In [ ]:
# Save cleaned and processed data to CSV files
monthly_levels.to_csv(
    processed_data_directory / "monthly_index_levels.csv",
)
monthly_returns_common.to_csv(
    processed_data_directory / "monthly_returns_common.csv",
)
monthly_returns_full.to_csv(
    processed_data_directory / "monthly_returns_full.csv",
)

## Output

The notebook should generate the following files:
- `monthly_index_levels.csv`
- `monthly_returns_full.csv`
- `monthly_returns_common.csv`

The generated files are excluded from the public repository due to data licensing restrictions.
